# Genie Space Configuration — Member Claims

This notebook is a **modular configuration tool** for managing the Genie space. Edit the configuration cells below, then run the execution cells to create or update the space.

## Notebook Structure
| Cell | Purpose | Action |
|---|---|---|
| **Cells 2-7** | Configuration (edit these) | Define space settings, instructions, metric views, questions, SQLs, benchmarks |
| **Cell 8** | Helper functions | Builds the `serialized_space` JSON from your config |
| **Cell 9** | Create / Update space | Calls the Genie API to apply changes |
| **Cell 10** | Validate space | Reads back the space and displays a summary |

## How to Use
1. Edit any configuration cell (Cells 2–7)
2. Run **Cell 8** (helpers), then **Cell 9** (apply), then **Cell 10** (validate)
3. To create a **new** space: set `SPACE_ID = ""` in Cell 2
4. To **update** an existing space: set `SPACE_ID = "<existing_id>"` in Cell 2

In [0]:
# =============================================================================
# SPACE CONFIGURATION
# =============================================================================

SPACE_TITLE = "Member Claims Analytics Genie - workshop"

SPACE_DESCRIPTION = (
    "Natural-language analytics for the Member Claims domain. "
    "Ask questions about claims costs, PMPM, member demographics, "
    "provider performance, denial rates, and utilization patterns. "
    "Powered by the Member Claims Metric View with MEASURE() syntax."
)

# Set to "" to CREATE a new space, or an existing ID to UPDATE
SPACE_ID = "01f13368688d1e229c8f19968aeed6a2"
WAREHOUSE_ID = "<your-sql-warehouse-id>"
PARENT_PATH = "/Workspace/Users/your.name@company.com/Customers/Member_Claims_Workshop/workshop_assets/members_claims_usecase/day2"

print(f"Space: {SPACE_TITLE}")
print(f"Mode:  {'UPDATE existing' if SPACE_ID else 'CREATE new'}")

Space: Member Claims Analytics Genie - workshop
Mode:  UPDATE existing


In [0]:
# =============================================================================
# GENERAL INSTRUCTIONS
# =============================================================================

GENERAL_INSTRUCTIONS = """
This Genie space provides natural-language access to the Member Claims Metric View.

## Metric View
- **Fully Qualified Name**: aira_test.aibi_workshop_mv_schema.member_claims_metric_view_workshop
- **Source**: fact_claim_detail (300,000 claim lines) joined with fact_claim_header and dim_member
- **Coverage**: Claims from April 2023 to March 2026

## Query Syntax
- Always wrap measures with MEASURE() function: `MEASURE(total_paid)`
- Always use GROUP BY ALL for grouped queries
- Use fully qualified metric view name in FROM clause

## Dimensions (for slicing/filtering)
- **service_month** (timestamp): Month of service, truncated to 1st of month
- **service_date** (date): Specific date of service
- **claim_type** (string): 'Institutional' or 'Professional'
- **line_of_business** (string): 'Commercial', 'Medicaid', 'Medicare', 'Medicare-Medicaid Plan', 'Exchange'
- **line_of_business_code** (string): Raw codes: COM, MCD, MCR, MMP, EXC
- **line_status** (string): 'Paid', 'Denied', 'Adjusted', 'Pending'
- **adjudication_status** (string): 'Final', 'Interim', 'Reopened'
- **clean_claim_indicator** (string): 'Y' or 'N'
- **par_provider_indicator** (string): 'Y' (in-network) or 'N' (out-of-network)
- **rendering_provider** (string): Provider name
- **provider_specialty** (string): Provider specialty
- **member_sex** (string): Member biological sex
- **member_state** (string): Member state of residence
- **member_zip_code** (string): Member ZIP code
- **member_age_group** (string): 'Under 18', '18-25', '26-35', '36-45', '46-55', '56-64', '65+'
- **member_line_of_business** (string): Member LOB from enrollment
- **benefit_category** (string): Benefit category
- **place_of_service** (string): Place of service code

## Core Measures (Atomic)
- **total_claims**: COUNT(DISTINCT claim_id) — distinct claim count (C-1)
- **total_claim_lines**: COUNT(*) — all service lines (C-2)
- **total_paid**: SUM(paid_amt) — total paid in USD (C-3). Always SUM, never LAST.
- **total_billed**: SUM(billed_amt) — total billed
- **total_allowed**: SUM(allowed_amt) — total allowed
- **unique_members**: COUNT(DISTINCT member_sk) — distinct members. Semi-additive, do NOT sum across time.
- **member_months**: COUNT(DISTINCT member-month) — denominator for PMPM

## Filtered Measures
- **denied_lines**: Claim lines with Denied status
- **clean_lines**: Claim lines flagged as clean
- **par_paid**: Paid amount for participating providers
- **inpatient_paid**: Paid for Institutional claims
- **outpatient_paid**: Paid for Professional claims
- **members_with_claims**: Members with at least one claim

## Composed Ratio Measures (NEVER SUM these)
- **avg_paid_per_claim**: total_paid / total_claims (C-4)
- **pmpm**: total_paid / member_months (MC-1). NEVER sum PMPM.
- **claims_per_1000_members**: (total_claims / member_months) * 1000 (MC-2)
- **denial_rate**: denied_lines / total_claim_lines (0 to 1)
- **clean_claim_rate**: clean_lines / total_claim_lines (0 to 1)
- **payment_to_billed_ratio**: total_paid / total_billed
- **payment_to_allowed_ratio**: total_paid / total_allowed
- **avg_paid_per_member**: total_paid / unique_members
- **claims_per_member**: total_claims / unique_members
- **lines_per_claim**: total_claim_lines / total_claims
- **par_provider_rate**: par_paid / total_paid

## Window Measures
- **trailing_3m_paid**: Paid over trailing 3-month window
- **trailing_3m_member_months**: Member-months over trailing 3-month window
- **rolling_3m_pmpm**: Smoothed PMPM over 3 months (W-1)
- **current_month_members**: Members in current month
- **previous_month_members**: Members in prior month
- **mom_active_member_growth**: Month-over-month member growth (W-2)

## Business Rules
- PMPM and all ratios must NEVER be summed across time — always recompute from components
- Members are semi-additive — do NOT sum across months
- Use NULLIF(denominator, 0) to guard against division by zero
- LOB values: Commercial, Medicaid, Medicare, Medicare-Medicaid Plan, Exchange
"""

print(f"Instructions length: {len(GENERAL_INSTRUCTIONS)} chars")

Instructions length: 3994 chars


In [0]:
# =============================================================================
# METRIC VIEW DESCRIPTIONS
# =============================================================================

METRIC_VIEW_DESCRIPTIONS = {
    "aira_test.aibi_workshop_mv_schema.member_claims_metric_view_workshop": (
        "Member Claims Metric View with 22 dimensions and 30 measures. "
        "Source: fact_claim_detail (300K rows) joined with fact_claim_header "
        "and dim_member. Covers claims from 2023-04 to 2026-03. "
        "Includes core claim measures (C-1 to C-4), PMPM (MC-1), "
        "claims per 1000 members (MC-2), denial rate, clean claim rate, "
        "payment ratios, provider analysis, and window measures "
        "(rolling 3M PMPM, MoM member growth). Dimensions include "
        "time, LOB, claim type, provider specialty, member demographics, "
        "and geography."
    ),
}

print(f"Metric views: {len(METRIC_VIEW_DESCRIPTIONS)}")
for k in sorted(METRIC_VIEW_DESCRIPTIONS):
    print(f"  * {k}")

Metric views: 1
  * aira_test.aibi_workshop_mv_schema.member_claims_metric_view_workshop


In [0]:
# =============================================================================
# SAMPLE QUESTIONS
# =============================================================================

SAMPLE_QUESTIONS = [
    # --- Financial Overview ---
    "What is the total paid amount across all claims?",
    "Show me the monthly PMPM trend",
    "Which line of business has the highest total cost?",
    "What is the rolling 3-month PMPM by month?",
    # --- Claims Analysis ---
    "What is the overall denial rate?",
    "Show me the clean claim rate trend by month",
    "What is the average paid per claim by claim type?",
    "How does payment-to-billed ratio vary by line of business?",
    # --- Member Demographics ---
    "How many unique members do we have by line of business?",
    "Show me member distribution by age group",
    "Which states have the highest total paid amount?",
    "What is the average paid per member by sex?",
    # --- Utilization & Provider ---
    "What are the claims per 1,000 members by LOB?",
    "Show inpatient vs outpatient paid by month",
    "What is the participating provider rate trend?",
    "Which provider specialties have the highest total cost?",
    # --- Window Measures ---
    "Show me the month-over-month member growth",
    "Compare monthly PMPM to rolling 3-month PMPM",
]

print(f"Sample questions: {len(SAMPLE_QUESTIONS)}")
for i, q in enumerate(SAMPLE_QUESTIONS, 1):
    print(f"  {i:2d}. {q}")

Sample questions: 18
   1. What is the total paid amount across all claims?
   2. Show me the monthly PMPM trend
   3. Which line of business has the highest total cost?
   4. What is the rolling 3-month PMPM by month?
   5. What is the overall denial rate?
   6. Show me the clean claim rate trend by month
   7. What is the average paid per claim by claim type?
   8. How does payment-to-billed ratio vary by line of business?
   9. How many unique members do we have by line of business?
  10. Show me member distribution by age group
  11. Which states have the highest total paid amount?
  12. What is the average paid per member by sex?
  13. What are the claims per 1,000 members by LOB?
  14. Show inpatient vs outpatient paid by month
  15. What is the participating provider rate trend?
  16. Which provider specialties have the highest total cost?
  17. Show me the month-over-month member growth
  18. Compare monthly PMPM to rolling 3-month PMPM


In [0]:
# =============================================================================
# EXAMPLE QUESTION SQLS
# =============================================================================

MV = "aira_test.aibi_workshop_mv_schema.member_claims_metric_view_workshop"

EXAMPLE_QUESTION_SQLS = [
    # --- Financial Overview ---
    (
        "What is the total paid amount?",
        f"SELECT MEASURE(total_paid) AS total_paid FROM {MV}"
    ),
    (
        "Show monthly PMPM trend",
        f"SELECT service_month, MEASURE(pmpm) AS pmpm FROM {MV} GROUP BY ALL ORDER BY service_month"
    ),
    (
        "Total paid by line of business",
        f"SELECT line_of_business, MEASURE(total_paid) AS total_paid FROM {MV} GROUP BY ALL ORDER BY total_paid DESC"
    ),
    (
        "Rolling 3-month PMPM by month",
        f"SELECT service_month, MEASURE(rolling_3m_pmpm) AS rolling_3m_pmpm FROM {MV} GROUP BY ALL ORDER BY service_month"
    ),
    # --- Claims Analysis ---
    (
        "What is the denial rate?",
        f"SELECT MEASURE(denial_rate) AS denial_rate FROM {MV}"
    ),
    (
        "Monthly denial rate and clean claim rate",
        f"SELECT service_month, MEASURE(denial_rate) AS denial_rate, MEASURE(clean_claim_rate) AS clean_claim_rate FROM {MV} GROUP BY ALL ORDER BY service_month"
    ),
    (
        "Average paid per claim by claim type",
        f"SELECT claim_type, MEASURE(avg_paid_per_claim) AS avg_paid_per_claim FROM {MV} GROUP BY ALL"
    ),
    (
        "Payment-to-billed ratio by LOB",
        f"SELECT line_of_business, MEASURE(payment_to_billed_ratio) AS pay_to_billed FROM {MV} GROUP BY ALL ORDER BY pay_to_billed DESC"
    ),
    # --- Member Demographics ---
    (
        "Unique members by line of business",
        f"SELECT line_of_business, MEASURE(unique_members) AS unique_members FROM {MV} GROUP BY ALL ORDER BY unique_members DESC"
    ),
    (
        "Member count by age group",
        f"SELECT member_age_group, MEASURE(unique_members) AS unique_members FROM {MV} GROUP BY ALL ORDER BY member_age_group"
    ),
    (
        "Total paid by state",
        f"SELECT member_state, MEASURE(total_paid) AS total_paid FROM {MV} GROUP BY ALL ORDER BY total_paid DESC LIMIT 15"
    ),
    (
        "Average paid per member by sex",
        f"SELECT member_sex, MEASURE(avg_paid_per_member) AS avg_paid_per_member FROM {MV} GROUP BY ALL"
    ),
    # --- Utilization & Provider ---
    (
        "Claims per 1,000 members by LOB",
        f"SELECT line_of_business, MEASURE(claims_per_1000_members) AS claims_per_1000 FROM {MV} GROUP BY ALL ORDER BY claims_per_1000 DESC"
    ),
    (
        "Inpatient vs outpatient paid by month",
        f"SELECT service_month, MEASURE(inpatient_paid) AS inpatient_paid, MEASURE(outpatient_paid) AS outpatient_paid FROM {MV} GROUP BY ALL ORDER BY service_month"
    ),
    (
        "Participating provider rate by month",
        f"SELECT service_month, MEASURE(par_provider_rate) AS par_rate FROM {MV} GROUP BY ALL ORDER BY service_month"
    ),
    (
        "Top specialties by total paid",
        f"SELECT provider_specialty, MEASURE(total_paid) AS total_paid, MEASURE(total_claims) AS total_claims FROM {MV} WHERE provider_specialty IS NOT NULL GROUP BY ALL ORDER BY total_paid DESC LIMIT 10"
    ),
    # --- Window Measures ---
    (
        "Month-over-month member growth",
        f"SELECT service_month, MEASURE(mom_active_member_growth) AS mom_growth FROM {MV} GROUP BY ALL ORDER BY service_month"
    ),
    (
        "PMPM vs rolling 3-month PMPM",
        f"SELECT service_month, MEASURE(pmpm) AS pmpm, MEASURE(rolling_3m_pmpm) AS rolling_3m_pmpm FROM {MV} GROUP BY ALL ORDER BY service_month"
    ),
]

print(f"Example SQLs: {len(EXAMPLE_QUESTION_SQLS)}")
for i, (q, _) in enumerate(EXAMPLE_QUESTION_SQLS, 1):
    print(f"  {i:2d}. {q}")

Example SQLs: 18
   1. What is the total paid amount?
   2. Show monthly PMPM trend
   3. Total paid by line of business
   4. Rolling 3-month PMPM by month
   5. What is the denial rate?
   6. Monthly denial rate and clean claim rate
   7. Average paid per claim by claim type
   8. Payment-to-billed ratio by LOB
   9. Unique members by line of business
  10. Member count by age group
  11. Total paid by state
  12. Average paid per member by sex
  13. Claims per 1,000 members by LOB
  14. Inpatient vs outpatient paid by month
  15. Participating provider rate by month
  16. Top specialties by total paid
  17. Month-over-month member growth
  18. PMPM vs rolling 3-month PMPM


In [0]:
# =============================================================================
# BENCHMARK QUESTIONS
# =============================================================================

BENCHMARK_QUESTIONS = [
    (
        "How much did we spend in total across all claims?",
        f"SELECT MEASURE(total_paid) AS total_paid FROM {MV}"
    ),
    (
        "What is the per-member-per-month cost trend?",
        f"SELECT service_month, MEASURE(pmpm) AS pmpm FROM {MV} GROUP BY ALL ORDER BY service_month"
    ),
    (
        "Break down total cost by insurance product",
        f"SELECT line_of_business, MEASURE(total_paid) AS total_paid FROM {MV} GROUP BY ALL ORDER BY total_paid DESC"
    ),
    (
        "Show me the smoothed 3-month PMPM over time",
        f"SELECT service_month, MEASURE(rolling_3m_pmpm) AS rolling_3m_pmpm FROM {MV} GROUP BY ALL ORDER BY service_month"
    ),
    (
        "What percentage of claims are denied?",
        f"SELECT MEASURE(denial_rate) AS denial_rate FROM {MV}"
    ),
    (
        "How has the clean claim percentage changed monthly?",
        f"SELECT service_month, MEASURE(clean_claim_rate) AS clean_claim_rate FROM {MV} GROUP BY ALL ORDER BY service_month"
    ),
    (
        "Compare average cost per claim for inpatient vs outpatient",
        f"SELECT claim_type, MEASURE(avg_paid_per_claim) AS avg_per_claim FROM {MV} GROUP BY ALL"
    ),
    (
        "What ratio of billed charges do we actually pay by product?",
        f"SELECT line_of_business, MEASURE(payment_to_billed_ratio) AS pay_to_billed FROM {MV} GROUP BY ALL"
    ),
    (
        "How many distinct members are in each insurance product?",
        f"SELECT line_of_business, MEASURE(unique_members) AS members FROM {MV} GROUP BY ALL"
    ),
    (
        "What does the age distribution look like for our members?",
        f"SELECT member_age_group, MEASURE(unique_members) AS member_count FROM {MV} GROUP BY ALL"
    ),
    (
        "Which geographic areas drive the most spending?",
        f"SELECT member_state, MEASURE(total_paid) AS total_paid FROM {MV} GROUP BY ALL ORDER BY total_paid DESC LIMIT 15"
    ),
    (
        "Is there a cost difference between male and female members?",
        f"SELECT member_sex, MEASURE(avg_paid_per_member) AS avg_cost FROM {MV} GROUP BY ALL"
    ),
    (
        "What is the claim frequency per thousand members by product?",
        f"SELECT line_of_business, MEASURE(claims_per_1000_members) AS claims_per_1000 FROM {MV} GROUP BY ALL"
    ),
    (
        "Show institutional vs professional spending over time",
        f"SELECT service_month, MEASURE(inpatient_paid) AS inpatient, MEASURE(outpatient_paid) AS outpatient FROM {MV} GROUP BY ALL ORDER BY service_month"
    ),
    (
        "Is our in-network provider utilization improving?",
        f"SELECT service_month, MEASURE(par_provider_rate) AS par_rate FROM {MV} GROUP BY ALL ORDER BY service_month"
    ),
    (
        "Which doctor specialties are most expensive?",
        f"SELECT provider_specialty, MEASURE(total_paid) AS total_paid FROM {MV} WHERE provider_specialty IS NOT NULL GROUP BY ALL ORDER BY total_paid DESC LIMIT 10"
    ),
    (
        "How is our membership base growing month to month?",
        f"SELECT service_month, MEASURE(mom_active_member_growth) AS growth FROM {MV} GROUP BY ALL ORDER BY service_month"
    ),
    (
        "How does monthly PMPM compare to the 3-month rolling average?",
        f"SELECT service_month, MEASURE(pmpm) AS monthly_pmpm, MEASURE(rolling_3m_pmpm) AS rolling_pmpm FROM {MV} GROUP BY ALL ORDER BY service_month"
    ),
]

print(f"Benchmark questions: {len(BENCHMARK_QUESTIONS)}")
for i, (q, _) in enumerate(BENCHMARK_QUESTIONS, 1):
    print(f"  {i:2d}. {q}")

Benchmark questions: 18
   1. How much did we spend in total across all claims?
   2. What is the per-member-per-month cost trend?
   3. Break down total cost by insurance product
   4. Show me the smoothed 3-month PMPM over time
   5. What percentage of claims are denied?
   6. How has the clean claim percentage changed monthly?
   7. Compare average cost per claim for inpatient vs outpatient
   8. What ratio of billed charges do we actually pay by product?
   9. How many distinct members are in each insurance product?
  10. What does the age distribution look like for our members?
  11. Which geographic areas drive the most spending?
  12. Is there a cost difference between male and female members?
  13. What is the claim frequency per thousand members by product?
  14. Show institutional vs professional spending over time
  15. Is our in-network provider utilization improving?
  16. Which doctor specialties are most expensive?
  17. How is our membership base growing month to month?

In [0]:
# =============================================================================
# HELPER FUNCTIONS
# Run this cell before Cell 9 (Create/Update) or Cell 10 (Validate).
# =============================================================================

import json
import uuid
import requests


def get_workspace_url():
    """Get the current workspace URL."""
    return spark.conf.get("spark.databricks.workspaceUrl")


def get_api_headers():
    """Get authorization headers for the Genie API."""
    token = (
        dbutils.notebook.entry_point.getDbutils()
        .notebook().getContext().apiToken().get()
    )
    return {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }


def _sorted_hex_ids(n: int) -> list[str]:
    """Generate *n* sorted 32-char lowercase hex UUIDs."""
    return sorted(uuid.uuid4().hex for _ in range(n))


def build_serialized_space(
    general_instructions: str,
    metric_view_descriptions: dict[str, str],
    sample_questions: list[str],
    example_question_sqls: list[tuple[str, str]],
    benchmark_questions: list[tuple[str, str]],
) -> str:
    """
    Build the serialized_space JSON string from user-editable config.

    Returns a JSON string ready for the Genie Space API.
    All IDs are auto-generated, sorted, and guaranteed unique.
    """

    # ---- Generate sorted IDs for every section ----
    sq_ids = _sorted_hex_ids(len(sample_questions))
    eq_ids = _sorted_hex_ids(len(example_question_sqls))
    bm_ids = _sorted_hex_ids(len(benchmark_questions))
    ti_id  = uuid.uuid4().hex                           # single instruction

    # Verify global uniqueness
    all_ids = sq_ids + eq_ids + bm_ids + [ti_id]
    assert len(all_ids) == len(set(all_ids)), "UUID collision — rerun the cell."

    # ---- config.sample_questions ----
    config_sq = [
        {"id": sq_ids[i], "question": [q]}
        for i, q in enumerate(sample_questions)
    ]

    # ---- data_sources.metric_views (sorted by identifier) ----
    mv_list = [
        {"identifier": k, "description": [v]}
        for k, v in sorted(metric_view_descriptions.items())
    ]

    # ---- instructions.text_instructions (exactly one entry) ----
    text_instr = [{"id": ti_id, "content": [general_instructions]}]

    # ---- instructions.example_question_sqls (sorted by id) ----
    ex_sqls = [
        {"id": eq_ids[i], "question": [q], "sql": [sql]}
        for i, (q, sql) in enumerate(example_question_sqls)
    ]

    # ---- benchmarks.questions (sorted by id) ----
    bm_list = [
        {
            "id": bm_ids[i],
            "question": [q],
            "answer": [{"format": "SQL", "content": [sql]}],
        }
        for i, (q, sql) in enumerate(benchmark_questions)
    ]

    # ---- Assemble full structure ----
    payload = {
        "version": 2,
        "config": {"sample_questions": config_sq},
        "data_sources": {"metric_views": mv_list},
        "instructions": {
            "text_instructions": text_instr,
            "example_question_sqls": ex_sqls,
        },
        "benchmarks": {"questions": bm_list},
    }

    return json.dumps(payload)


print("✅ Helper functions loaded: get_workspace_url, get_api_headers, build_serialized_space")

✅ Helper functions loaded: get_workspace_url, get_api_headers, build_serialized_space


In [0]:
# =============================================================================
# CREATE OR UPDATE GENIE SPACE
# Run this cell to apply the configuration from Cells 2–7.
# =============================================================================

serialised = build_serialized_space(
    general_instructions=GENERAL_INSTRUCTIONS,
    metric_view_descriptions=METRIC_VIEW_DESCRIPTIONS,
    sample_questions=SAMPLE_QUESTIONS,
    example_question_sqls=EXAMPLE_QUESTION_SQLS,
    benchmark_questions=BENCHMARK_QUESTIONS,
)

ws_url  = get_workspace_url()
headers = get_api_headers()

if SPACE_ID:
    # ---------- UPDATE existing space ----------
    print(f"Updating space {SPACE_ID} ...")
    resp = requests.patch(
        f"https://{ws_url}/api/2.0/genie/spaces/{SPACE_ID}",
        headers=headers,
        json={
            "title": SPACE_TITLE,
            "description": SPACE_DESCRIPTION,
            "serialized_space": serialised,
        },
    )
else:
    # ---------- CREATE new space ----------
    print("Creating new space ...")
    resp = requests.post(
        f"https://{ws_url}/api/2.0/genie/spaces",
        headers=headers,
        json={
            "title": SPACE_TITLE,
            "description": SPACE_DESCRIPTION,
            "warehouse_id": WAREHOUSE_ID,
            "parent_path": PARENT_PATH,
            "serialized_space": serialised,
        },
    )

# ---------- Report result ----------
if resp.status_code == 200:
    result = resp.json()
    new_id = result.get("space_id", SPACE_ID)
    print(f"\n✅ SUCCESS")
    print(f"   Space ID   : {new_id}")
    print(f"   Title       : {result.get('title')}")
    print(f"   Description : {result.get('description', '')[:120]}...")
    if not SPACE_ID:
        print(f"\n⚠️  Copy this ID into Cell 2 for future updates:")
        print(f'   SPACE_ID = "{new_id}"')
else:
    print(f"\n❌ FAILED ({resp.status_code})")
    err = resp.json()
    print(f"   Error : {err.get('error_code')}")
    print(f"   Message: {err.get('message', '')[:300]}")

Creating new space ...

✅ SUCCESS
   Space ID   : 01f13368688d1e229c8f19968aeed6a2
   Title       : Member Claims Analytics Genie - workshop
   Description : Natural-language analytics for the Member Claims domain. Ask questions about claims costs, PMPM, member demographics, pr...

⚠️  Copy this ID into Cell 2 for future updates:
   SPACE_ID = "01f13368688d1e229c8f19968aeed6a2"


In [0]:
# =============================================================================
# VALIDATE SPACE
# Read back the space config and display a summary.
# =============================================================================

target_id = SPACE_ID
if not target_id:
    print("No SPACE_ID set — set it in Cell 2 after creating a space.")
else:
    ws_url  = get_workspace_url()
    headers = get_api_headers()

    resp = requests.get(
        f"https://{ws_url}/api/2.0/genie/spaces/{target_id}"
        "?include_serialized_space=true",
        headers=headers,
    )

    if resp.status_code != 200:
        print(f"❌ Failed to read space: {resp.status_code}")
    else:
        data = resp.json()
        ss   = json.loads(data["serialized_space"])

        sqs  = ss.get("config", {}).get("sample_questions", [])
        mvs  = ss.get("data_sources", {}).get("metric_views", [])
        tis  = ss.get("instructions", {}).get("text_instructions", [])
        eqs  = ss.get("instructions", {}).get("example_question_sqls", [])
        bms  = ss.get("benchmarks", {}).get("questions", [])

        print("=" * 60)
        print("GENIE SPACE VALIDATION REPORT")
        print("=" * 60)
        print(f"Title        : {data.get('title')}")
        print(f"Space ID     : {data.get('space_id', target_id)}")
        print(f"Warehouse    : {data.get('warehouse_id', 'N/A')}")
        print(f"Description  : {data.get('description', '')[:120]}...")
        print()
        print(f"Metric Views        : {len(mvs)}")
        for mv in mvs:
            print(f"   • {mv['identifier'].split('.')[-1]}")
        print(f"Sample Questions    : {len(sqs)}")
        print(f"Example SQLs        : {len(eqs)}")
        print(f"Benchmark Questions : {len(bms)}")
        print(f"Text Instructions   : {len(tis)} block(s), "
              f"{sum(len(t['content'][0]) for t in tis)} chars")
        print()

        if bms:
            print("-" * 60)
            print("BENCHMARK QUESTIONS")
            print("-" * 60)
            for i, bm in enumerate(bms, 1):
                q = bm['question'][0]
                has_sql = bool(bm.get('answer'))
                marker = "✅" if has_sql else "⚠️"
                print(f"  {i:2d}. {marker} {q}")

        print()
        print("✅ Validation complete")

GENIE SPACE VALIDATION REPORT
Title        : Member Claims Analytics Genie - workshop
Space ID     : 01f13368688d1e229c8f19968aeed6a2
Warehouse    : <your-sql-warehouse-id>
Description  : Natural-language analytics for the Member Claims domain. Ask questions about claims costs, PMPM, member demographics, pr...

Metric Views        : 1
   • member_claims_metric_view_workshop
Sample Questions    : 18
Example SQLs        : 18
Benchmark Questions : 18
Text Instructions   : 1 block(s), 1 chars

------------------------------------------------------------
BENCHMARK QUESTIONS
------------------------------------------------------------
   1. ✅ How does monthly PMPM compare to the 3-month rolling average?
   2. ✅ How is our membership base growing month to month?
   3. ✅ Which doctor specialties are most expensive?
   4. ✅ Is our in-network provider utilization improving?
   5. ✅ Show institutional vs professional spending over time
   6. ✅ What is the claim frequency per thousand members by p